# Лабораторная работа 6: Реализация FGSM атаки и разработка и анализ эффективности защитных мер


## Описание лабораторной работы:

FGSM является adversarial атакой (MITRE ATLAS TML0014: Adversarial Attack)

Цели работы

1.	Изучить принцип работы adversarial-атак на модели машинного обучения.

2.	Реализовать атаку Fast Gradient Sign Method (FGSM) на нейронную сеть.

3.	Количественно оценить влияние adversarial-примеров на точность модели.

4.	Разработать и протестировать защитные меры (Adversarial Training, Input Preprocessing).

5.	Проанализировать эффективность защиты и компромиссы между безопасностью и производительностью.


In [1]:
import torch

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
import os
from PIL import Image

# Для воспроизводимости
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Описание модели нейронной сети (CNN для распознавания MNIST/CIFAR-10)

In [2]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10, input_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7 if input_channels == 1 else 64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def predict(self, x):
        self.eval()
        with torch.no_grad():
            outputs = self(x)
            _, predicted = torch.max(outputs, 1)
        return predicted

## Реализация FGSM атаки

In [3]:
class FGSMAttack:
    """Реализация Fast Gradient Sign Method атаки"""

    def __init__(self, model, epsilon=0.1):
        self.model = model
        self.epsilon = epsilon
        self.criterion = nn.CrossEntropyLoss()

    def attack(self, data, target):
        """
        Создаёт adversarial-пример для одного изображения

        Args:
            data: Тензор изображения [1, C, H, W]
            target: Истинная метка класса

        Returns:
            adversarial_data: Adversarial-пример
            perturbation: Добавленное возмущение
        """
        self.model.eval()
        data.requires_grad = True

        # Forward pass
        output = self.model(data)
        loss = self.criterion(output, target.unsqueeze(0))

        # Backward pass для получения градиента по входным данным
        loss.backward()

        # Получаем знак градиента
        data_grad = data.grad.data.sign()

        # Создаём adversarial-пример
        perturbation = self.epsilon * data_grad
        adversarial_data = data + perturbation

        # Ограничиваем значения в диапазоне [0, 1]
        adversarial_data = torch.clamp(adversarial_data, 0, 1)

        # Очищаем градиенты
        data.grad = None

        return adversarial_data.detach(), perturbation.detach()

    def attack_batch(self, dataloader, device='cpu'):
        """
        Применяет атаку ко всему набору данных

        Returns:
            original_images: Оригинальные изображения
            adversarial_images: Adversarial-изображения
            original_labels: Истинные метки
            original_predictions: Предсказания на оригинальных данных
            adversarial_predictions: Предсказания на adversarial-данных
            perturbations: Возмущения
        """
        self.model.eval()

        original_images = []
        adversarial_images = []
        original_labels = []
        original_predictions = []
        adversarial_predictions = []
        perturbations = []

        print(f"Применение FGSM атаки (epsilon={self.epsilon})")

        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(device), target.to(device)

            # Оригинальное предсказание
            orig_output = self.model(data)
            _, orig_pred = torch.max(orig_output, 1)

            # Атака на каждый образец в батче
            adv_batch = []
            adv_pred_batch = []
            pert_batch = []

            for i in range(len(data)):
                adv_data, pert = self.attack(data[i:i+1], target[i])
                adv_batch.append(adv_data)
                pert_batch.append(pert)

            adv_batch = torch.cat(adv_batch, dim=0)
            pert_batch = torch.cat(pert_batch, dim=0)

            # Adversarial предсказание
            adv_output = self.model(adv_batch)
            _, adv_pred = torch.max(adv_output, 1)

            # Сохранение результатов
            original_images.append(data.cpu())
            adversarial_images.append(adv_batch.cpu())
            original_labels.append(target.cpu())
            original_predictions.append(orig_pred.cpu())
            adversarial_predictions.append(adv_pred.cpu())
            perturbations.append(pert_batch.cpu())

            if (batch_idx + 1) % 10 == 0:
                print(f"  Обработано батчей: {batch_idx + 1}")

        # Конкатенация всех батчей
        return (
            torch.cat(original_images, dim=0),
            torch.cat(adversarial_images, dim=0),
            torch.cat(original_labels, dim=0),
            torch.cat(original_predictions, dim=0),
            torch.cat(adversarial_predictions, dim=0),
            torch.cat(perturbations, dim=0)
        )

## Реализация защитных мер

In [4]:
class DefenseMeasures:
    """Реализация защитных мер против adversarial-атак"""

    def adversarial_training(model, dataloader, epsilon=0.1, epochs=5, device='cpu'):
        """
        Adversarial Training: обучение модели на adversarial-примерах
        """
        print(f"\nAdversarial Training (epsilon={epsilon}, epochs={epochs})...")

        model.train()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        history = {'train_loss': [], 'train_acc': []}

        for epoch in range(epochs):
            total_loss = 0
            correct = 0
            total = 0

            for batch_idx, (data, target) in enumerate(dataloader):
                data, target = data.to(device), target.to(device)

                # 1. Обучение на оригинальных данных
                optimizer.zero_grad()
                output = model(data)
                loss_orig = criterion(output, target)
                loss_orig.backward()

                # 2. Создание adversarial-примеров
                data.requires_grad = True
                output = model(data)
                loss = criterion(output, target)
                loss.backward()

                data_grad = data.grad.data.sign()
                adv_data = data + epsilon * data_grad
                adv_data = torch.clamp(adv_data, 0, 1)

                # 3. Обучение на adversarial-примерах
                optimizer.zero_grad()
                output_adv = model(adv_data)
                loss_adv = criterion(output_adv, target)
                loss_adv.backward()

                optimizer.step()

                total_loss += loss_adv.item()
                _, predicted = torch.max(output_adv, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

            avg_loss = total_loss / len(dataloader)
            accuracy = correct / total
            history['train_loss'].append(avg_loss)
            history['train_acc'].append(accuracy)

            print(f"  Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {accuracy:.4f}")

        return model, history

    def input_preprocessing(image, method='median_smoothing', kernel_size=3):
        """
        Предобработка входных данных для удаления шумов

        Methods:
            - median_smoothing: Медианная фильтрация
            - gaussian_blur: Размытие по Гауссу
            - bit_depth_reduction: Уменьшение глубины цвета
        """
        import torch.nn.functional as F

        if method == 'median_smoothing':
            # Простая реализация через усреднение
            padding = kernel_size // 2
            image_padded = F.pad(image, (padding, padding, padding, padding), mode='reflect')
            unfolded = F.unfold(image_padded, kernel_size=kernel_size)
            smoothed = unfolded.median(dim=1, keepdim=True)[0]
            smoothed = F.fold(smoothed, output_size=image.shape[2:], kernel_size=1)
            return smoothed

        elif method == 'gaussian_blur':
            # Простое размытие через усреднение
            kernel = torch.ones(1, 1, kernel_size, kernel_size) / (kernel_size ** 2)
            kernel = kernel.to(image.device)
            smoothed = F.conv2d(image, kernel, padding=kernel_size//2, groups=image.shape[1])
            return smoothed

        elif method == 'bit_depth_reduction':
            # Квантование
            bits = 4
            max_val = 2 ** bits - 1
            quantized = torch.round(image * max_val) / max_val
            return quantized

        return image

    def detect_adversarial(image, original_image, threshold=0.05):
        """
        Обнаружение adversarial-примеров по величине возмущения
        """
        perturbation = torch.abs(image - original_image)
        mean_perturbation = perturbation.mean()

        is_adversarial = mean_perturbation > threshold
        confidence = float(mean_perturbation)

        return is_adversarial.item(), confidence

## Функции для визуализации результатов

In [5]:
class FGSMVisualizer:
    """Визуализация результатов FGSM атаки и защиты"""

    def __init__(self, save_dir="results"):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

    def plot_adversarial_examples(self, original, adversarial, labels, pred_orig, pred_adv,
                                  num_samples=5, title="Визуализация оригинальных и adversarial-изображений"):
        plt.figure(figsize=(20, 4))

        for i in range(num_samples):
            # Оригинальное
            plt.subplot(3, num_samples, i + 1)
            img = original[i].squeeze().cpu().numpy()
            plt.imshow(img, cmap='gray' if img.shape[0] == 1 else None)
            plt.title(f"Orig: {labels[i]}\nPred: {pred_orig[i]}")
            plt.axis('off')

            # Adversarial
            plt.subplot(3, num_samples, i + 1 + num_samples)
            img = adversarial[i].squeeze().cpu().numpy()
            plt.imshow(img, cmap='gray' if img.shape[0] == 1 else None)
            plt.title(f"Adv: {labels[i]}\nPred: {pred_adv[i]}")
            plt.axis('off')

            # Возмущение (усиленное для видимости)
            plt.subplot(3, num_samples, i + 1 + 2*num_samples)
            pert = (adversarial[i] - original[i]).squeeze().cpu().numpy()
            pert_normalized = (pert - pert.min()) / (pert.max() - pert.min() + 1e-8)
            plt.imshow(pert_normalized, cmap='coolwarm')
            plt.title(f"Возмущение (perturbation) (x10)")
            plt.axis('off')

        plt.suptitle(title, fontsize=16)
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/adversarial_examples.png", dpi=300)
        plt.close()
        print(f"Сохранено: {self.save_dir}/adversarial_examples.png")

    def plot_accuracy_vs_epsilon(self, results_df, title="График точности (Accuracy) в зависимости от epsilon"):
        plt.figure(figsize=(10, 6))

        plt.plot(results_df['epsilon'], results_df['original_accuracy'],
                'o-', label='Original Accuracy', linewidth=2, markersize=8)
        plt.plot(results_df['epsilon'], results_df['adversarial_accuracy'],
                's--', label='Adversarial Accuracy', linewidth=2, markersize=8)
        plt.plot(results_df['epsilon'], results_df['defended_accuracy'],
                '^-', label='Defended Accuracy', linewidth=2, markersize=8)

        plt.xlabel('Epsilon (ε)')
        plt.ylabel('Accuracy')
        plt.title(title)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(results_df['epsilon'])

        plt.savefig(f"{self.save_dir}/accuracy_vs_epsilon.png", dpi=300)
        plt.close()
        print(f"Сохранено: {self.save_dir}/accuracy_vs_epsilon.png")

    def plot_confusion_matrix(self, y_true, y_pred, title="Матрица ошибок (Confusion Matrix)"):
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title(title)
        plt.savefig(f"{self.save_dir}/confusion_matrix.png", dpi=300)
        plt.close()
        print(f"Сохранено: {self.save_dir}/confusion_matrix.png")

    def plot_perturbation_distribution(self, perturbations, title="Распределение возмущений (Perturbation Distribution)"):
        plt.figure(figsize=(12, 5))

        pert_magnitudes = perturbations.abs().mean(dim=[1, 2, 3]).cpu().numpy()

        plt.subplot(1, 2, 1)
        plt.hist(pert_magnitudes, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
        plt.xlabel('Mean Perturbation Magnitude')
        plt.ylabel('Frequency')
        plt.title(f'{title}\nMean={pert_magnitudes.mean():.4f}')
        plt.grid(True, alpha=0.3)

        plt.subplot(1, 2, 2)
        sns.boxplot(data=pert_magnitudes)
        plt.ylabel('Perturbation Magnitude')
        plt.title('Box Plot')

        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/perturbation_distribution.png", dpi=300)
        plt.close()
        print(f"Сохранено: {self.save_dir}/perturbation_distribution.png")

## ОСНОВНОЙ СЦЕНАРИЙ ЛАБОРАТОРНОЙ РАБОТЫ

In [6]:
print("\n" + "="*70)
print("ЛАБОРАТОРНАЯ РАБОТА: FGSM ADVERSARIAL ATTACK И ЗАЩИТА")
print("MITRE ATLAS TML0014: Adversarial Attack")
print("="*70)


ЛАБОРАТОРНАЯ РАБОТА: FGSM ADVERSARIAL ATTACK И ЗАЩИТА
MITRE ATLAS TML0014: Adversarial Attack


In [7]:
# Параметры
BATCH_SIZE = 64
NUM_CLASSES = 10
EPSILONS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nИспользуемое устройство: {DEVICE}")

visualizer = FGSMVisualizer(save_dir="results")
results_summary = []



Используемое устройство: cuda


# 1. Загрузка данных (MNIST)

In [8]:
print("\n" + "="*70)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ (MNIST)")
print("="*70)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Загрузка тестового набора для атаки
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Загрузка тренировочного набора для обучения защиты
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Загружено тестовых образцов: {len(test_dataset)}")
print(f"Загружено тренировочных образцов: {len(train_dataset)}")


ШАГ 1: ЗАГРУЗКА ДАННЫХ (MNIST)


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 435kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.11MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]

Загружено тестовых образцов: 10000
Загружено тренировочных образцов: 60000


# 2. Обучение базовой модели

In [9]:
print("\n" + "="*70)
print("ШАГ 2: ОБУЧЕНИЕ БАЗОВОЙ МОДЕЛИ")
print("="*70)

model = SimpleCNN(num_classes=NUM_CLASSES, input_channels=1).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Обучение (5 эпох для демонстрации)
model.train()
for epoch in range(5):
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"  Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.4f}")

# Оценка на тесте
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        output = model(data)
        _, predicted = torch.max(output, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

baseline_accuracy = correct / total
print(f"\nБазовая точность модели: {baseline_accuracy:.4f}")


ШАГ 2: ОБУЧЕНИЕ БАЗОВОЙ МОДЕЛИ
  Epoch 1/5 | Loss: 0.2115
  Epoch 2/5 | Loss: 0.0790
  Epoch 3/5 | Loss: 0.0619
  Epoch 4/5 | Loss: 0.0509
  Epoch 5/5 | Loss: 0.0415

Базовая точность модели: 0.9909


# 3. FGSM Атака с разными $\epsilon$

In [10]:
print("\n" + "="*70)
print("ШАГ 3: FGSM АТАКА С РАЗНЫМИ EPSILON")
print("="*70)

for epsilon in EPSILONS:
    print(f"\nАтака с epsilon={epsilon}")

    attacker = FGSMAttack(model, epsilon=epsilon)

    orig_imgs, adv_imgs, labels, orig_preds, adv_preds, perts = attacker.attack_batch(
        test_loader, device=DEVICE
    )

    # Метрики
    orig_accuracy = accuracy_score(labels.cpu(), orig_preds.cpu())
    adv_accuracy = accuracy_score(labels.cpu(), adv_preds.cpu())
    success_rate = 1 - adv_accuracy / orig_accuracy if orig_accuracy > 0 else 0

    print(f"   Original Accuracy: {orig_accuracy:.4f}")
    print(f"   Adversarial Accuracy: {adv_accuracy:.4f}")
    print(f"   Attack Success Rate: {success_rate:.4f}")

    # Сохранение результатов для epsilon=0.1 (для визуализации)
    if epsilon == 0.1:
        visualizer.plot_adversarial_examples(
            orig_imgs[:10], adv_imgs[:10], labels[:10],
            orig_preds[:10], adv_preds[:10],
            num_samples=5,
            title=f"FGSM Attack Examples (ε={epsilon})"
        )
        visualizer.plot_perturbation_distribution(perts, title=f"Perturbation Distribution (ε={epsilon})")
        visualizer.plot_confusion_matrix(labels.cpu(), adv_preds.cpu(),
                                       title=f"Confusion Matrix (ε={epsilon})")

    results_summary.append({
        'epsilon': epsilon,
        'original_accuracy': orig_accuracy,
        'adversarial_accuracy': adv_accuracy,
        'attack_success_rate': success_rate,
        'mean_perturbation': float(perts.abs().mean()),
        'defended_accuracy': None  # Будет заполнено позже
    })


ШАГ 3: FGSM АТАКА С РАЗНЫМИ EPSILON

Атака с epsilon=0.0
Применение FGSM атаки (epsilon=0.0)
  Обработано батчей: 10
  Обработано батчей: 20
  Обработано батчей: 30
  Обработано батчей: 40
  Обработано батчей: 50
  Обработано батчей: 60
  Обработано батчей: 70
  Обработано батчей: 80
  Обработано батчей: 90
  Обработано батчей: 100
  Обработано батчей: 110
  Обработано батчей: 120
  Обработано батчей: 130
  Обработано батчей: 140
  Обработано батчей: 150
   Original Accuracy: 0.9909
   Adversarial Accuracy: 0.9866
   Attack Success Rate: 0.0043

Атака с epsilon=0.05
Применение FGSM атаки (epsilon=0.05)
  Обработано батчей: 10
  Обработано батчей: 20
  Обработано батчей: 30
  Обработано батчей: 40
  Обработано батчей: 50
  Обработано батчей: 60
  Обработано батчей: 70
  Обработано батчей: 80
  Обработано батчей: 90
  Обработано батчей: 100
  Обработано батчей: 110
  Обработано батчей: 120
  Обработано батчей: 130
  Обработано батчей: 140
  Обработано батчей: 150
   Original Accuracy: 0

# 4. Применение защиты (Adversarial Training)

In [11]:
print("\n" + "="*70)
print("ШАГ 4: ПРИМЕНЕНИЕ ЗАЩИТЫ (ADVERSARIAL TRAINING)")
print("="*70)

# Переобучение модели с adversarial training
model_defended = SimpleCNN(num_classes=NUM_CLASSES, input_channels=1).to(DEVICE)
model_defended, train_history = DefenseMeasures.adversarial_training(
    model_defended, train_loader, epsilon=0.1, epochs=5, device=DEVICE
)

# Тестирование защищённой модели
print("\nТестирование защищённой модели...")

for i, epsilon in enumerate(EPSILONS):
    attacker = FGSMAttack(model_defended, epsilon=epsilon)

    orig_imgs, adv_imgs, labels, orig_preds, adv_preds, perts = attacker.attack_batch(
        test_loader, device=DEVICE
    )

    defended_accuracy = accuracy_score(labels.cpu(), adv_preds.cpu())
    results_summary[i]['defended_accuracy'] = defended_accuracy

    print(f"   Epsilon={epsilon:.2f} | Defended Accuracy: {defended_accuracy:.4f}")


ШАГ 4: ПРИМЕНЕНИЕ ЗАЩИТЫ (ADVERSARIAL TRAINING)

Adversarial Training (epsilon=0.1, epochs=5)...
  Epoch 1/5 | Loss: 0.3172 | Acc: 0.9008
  Epoch 2/5 | Loss: 0.1185 | Acc: 0.9653
  Epoch 3/5 | Loss: 0.0914 | Acc: 0.9730
  Epoch 4/5 | Loss: 0.0731 | Acc: 0.9784
  Epoch 5/5 | Loss: 0.0638 | Acc: 0.9808

Тестирование защищённой модели...
Применение FGSM атаки (epsilon=0.0)
  Обработано батчей: 10
  Обработано батчей: 20
  Обработано батчей: 30
  Обработано батчей: 40
  Обработано батчей: 50
  Обработано батчей: 60
  Обработано батчей: 70
  Обработано батчей: 80
  Обработано батчей: 90
  Обработано батчей: 100
  Обработано батчей: 110
  Обработано батчей: 120
  Обработано батчей: 130
  Обработано батчей: 140
  Обработано батчей: 150
   Epsilon=0.00 | Defended Accuracy: 0.9895
Применение FGSM атаки (epsilon=0.05)
  Обработано батчей: 10
  Обработано батчей: 20
  Обработано батчей: 30
  Обработано батчей: 40
  Обработано батчей: 50
  Обработано батчей: 60
  Обработано батчей: 70
  Обработан

# 5. Сводные результаты

In [12]:
print("\n" + "="*70)
print("ШАГ 5: СВОДНЫЕ РЕЗУЛЬТАТЫ")
print("="*70)

df_results = pd.DataFrame(results_summary)
print(df_results.to_string(index=False))

# Сохранение результатов
with open("results/summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

df_results.to_csv("results/summary.csv", index=False)
print(f"\nРезультаты сохранены в results/")

# Визуализация
visualizer.plot_accuracy_vs_epsilon(df_results, title="FGSM атака: Accuracy vs Epsilon")


ШАГ 5: СВОДНЫЕ РЕЗУЛЬТАТЫ
 epsilon  original_accuracy  adversarial_accuracy  attack_success_rate  mean_perturbation  defended_accuracy
    0.00             0.9909                0.9866             0.004339           0.000000             0.9895
    0.05             0.9909                0.9842             0.006762           0.043137             0.9877
    0.10             0.9909                0.9820             0.008982           0.086273             0.9860
    0.15             0.9909                0.9804             0.010596           0.129410             0.9839
    0.20             0.9909                0.9764             0.014633           0.172547             0.9823
    0.25             0.9909                0.9721             0.018973           0.215684             0.9794
    0.30             0.9909                0.9673             0.023817           0.258820             0.9768

Результаты сохранены в results/
Сохранено: results/accuracy_vs_epsilon.png


# 6. Анализ эффективности защиты

In [13]:
print("\n" + "="*70)
print("АНАЛИЗ ЭФФЕКТИВНОСТИ ЗАЩИТЫ")
print("="*70)

for epsilon in EPSILONS:
    row = df_results[df_results['epsilon'] == epsilon].iloc[0]
    improvement = row['defended_accuracy'] - row['adversarial_accuracy']
    print(f"ε={epsilon:.2f}: Улучшение защиты = {improvement:.4f} "
          f"({row['adversarial_accuracy']:.4f} → {row['defended_accuracy']:.4f})")



АНАЛИЗ ЭФФЕКТИВНОСТИ ЗАЩИТЫ
ε=0.00: Улучшение защиты = 0.0029 (0.9866 → 0.9895)
ε=0.05: Улучшение защиты = 0.0035 (0.9842 → 0.9877)
ε=0.10: Улучшение защиты = 0.0040 (0.9820 → 0.9860)
ε=0.15: Улучшение защиты = 0.0035 (0.9804 → 0.9839)
ε=0.20: Улучшение защиты = 0.0059 (0.9764 → 0.9823)
ε=0.25: Улучшение защиты = 0.0073 (0.9721 → 0.9794)
ε=0.30: Улучшение защиты = 0.0095 (0.9673 → 0.9768)


In [14]:
df_results.head(3)

,epsilon,original_accuracy,adversarial_accuracy,attack_success_rate,mean_perturbation,defended_accuracy
0,0.00,0.9909,0.9866,0.004339,0.000000,0.9895
1,0.05,0.9909,0.9842,0.006762,0.043137,0.9877
2,0.10,0.9909,0.9820,0.008982,0.086273,0.9860


# References